# FraudShield AI — Notebook 02: Data Preprocessing

This notebook:
- Loads and cleans the dataset
- Encodes categorical variables
- Engineers basic features safely (without target leakage)
- Creates a chronological train/test split
- Saves processed datasets for model training

In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
import joblib
warnings.filterwarnings('ignore')

DATA_PATH  = '../data/raw/transactions.csv'
PROC_PATH  = '../data/processed/'
MODEL_PATH = '../models/'
os.makedirs(PROC_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f'Dataset not found at {DATA_PATH}.\n'
        'Please run 01_EDA.ipynb first to generate the demo dataset, '
        'or place the real PaySim CSV at data/raw/transactions.csv'
    )

df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df):,} rows × {len(df.columns)} columns')
df.head(3)

In [ ]:
# ── Column Mapping ────────────────────────────────────────────────
# Standardize column names for consistency
rename_map = {
    'type':          'transaction_type',
    'oldbalanceOrg': 'origin_balance_before',
    'newbalanceOrig': 'origin_balance_after',
    'oldbalanceDest': 'dest_balance_before',
    'newbalanceDest': 'dest_balance_after',
    'nameOrig':      'origin_account',
    'nameDest':      'destination_account',
    'isFraud':       'is_fraud',
}
df.rename(columns=rename_map, inplace=True)

# Drop columns that cause target leakage or are not available at prediction time
leakage_cols = ['isFlaggedFraud']  # This is a downstream system flag, not available pre-fraud
df.drop(columns=[c for c in leakage_cols if c in df.columns], inplace=True)
print('Columns after cleanup:', df.columns.tolist())

In [ ]:
# ── Missing Values ────────────────────────────────────────────────
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values — dataset is complete.')
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
print(f'After cleanup: {len(df):,} rows')

In [ ]:
# ── Encode Transaction Type ───────────────────────────────────────
TYPE_MAP = {'CASH_IN': 0, 'CASH_OUT': 1, 'DEBIT': 2, 'PAYMENT': 3, 'TRANSFER': 4}
df['transaction_type_encoded'] = df['transaction_type'].map(TYPE_MAP)
print('Transaction type encoding:')
print(df[['transaction_type','transaction_type_encoded']].drop_duplicates().sort_values('transaction_type_encoded'))

In [ ]:
# ── Feature Engineering (safe, no leakage) ───────────────────────
# These features use only data available at transaction time
df['balance_diff_origin']    = df['origin_balance_before'] - df['origin_balance_after']
df['balance_diff_dest']      = df['dest_balance_after']    - df['dest_balance_before']
df['amount_to_origin_ratio'] = df['amount'] / (df['origin_balance_before'] + 1e-9)
df['is_large_transfer']      = (df['amount'] > 200_000).astype(int)

print('Engineered features added:')
print(['balance_diff_origin', 'balance_diff_dest', 'amount_to_origin_ratio', 'is_large_transfer'])

In [ ]:
# ── Feature Selection ─────────────────────────────────────────────
FEATURE_COLS = [
    'transaction_type_encoded',
    'amount',
    'origin_balance_before',
    'origin_balance_after',
    'dest_balance_before',
    'dest_balance_after',
    'balance_diff_origin',
    'balance_diff_dest',
    'amount_to_origin_ratio',
    'is_large_transfer',
]
TARGET_COL = 'is_fraud'

print(f'Features: {FEATURE_COLS}')
print(f'Target:   {TARGET_COL}')

In [ ]:
# ── Chronological Train/Test Split ────────────────────────────────
# If 'step' (time) column exists, sort by it and use the last 20% as test
# This prevents future leakage (a critical mistake in fraud models)
if 'step' in df.columns:
    df.sort_values('step', inplace=True)
    df.reset_index(drop=True, inplace=True)
    split_idx = int(len(df) * 0.80)
    train_df = df.iloc[:split_idx]
    test_df  = df.iloc[split_idx:]
    print(f'Chronological split (time-based):')
else:
    # Random split if no time info
    from sklearn.model_selection import train_test_split
    train_df, test_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df[TARGET_COL])
    print(f'Random stratified split:')

print(f'  Train: {len(train_df):,} rows | Fraud: {train_df[TARGET_COL].sum():,} ({train_df[TARGET_COL].mean()*100:.2f}%)')
print(f'  Test:  {len(test_df):,}  rows | Fraud: {test_df[TARGET_COL].sum():,}  ({test_df[TARGET_COL].mean()*100:.2f}%)')

In [ ]:
# ── Fit Scaler on TRAINING DATA ONLY ─────────────────────────────
# NEVER fit on test data — this is a common source of data leakage
X_train = train_df[FEATURE_COLS].values
y_train = train_df[TARGET_COL].values
X_test  = test_df[FEATURE_COLS].values
y_test  = test_df[TARGET_COL].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)         # transform only on test

print(f'Scaler fitted on training data.')
print(f'X_train shape: {X_train_scaled.shape}')
print(f'X_test shape:  {X_test_scaled.shape}')

In [ ]:
# ── Save Processed Data & Preprocessor ───────────────────────────
# Save train/test as numpy arrays
np.save(f'{PROC_PATH}X_train.npy', X_train_scaled)
np.save(f'{PROC_PATH}y_train.npy', y_train)
np.save(f'{PROC_PATH}X_test.npy',  X_test_scaled)
np.save(f'{PROC_PATH}y_test.npy',  y_test)

# Also save unscaled for XGBoost (which doesn't need scaling)
np.save(f'{PROC_PATH}X_train_raw.npy', X_train)
np.save(f'{PROC_PATH}X_test_raw.npy',  X_test)

# Save feature column order
import json
with open(f'{PROC_PATH}feature_columns.json', 'w') as f:
    json.dump(FEATURE_COLS, f)

# Save scaler as the preprocessor
joblib.dump(scaler, f'{MODEL_PATH}preprocessor.joblib')

print('Saved:')
print(f'  {PROC_PATH}X_train.npy, y_train.npy, X_test.npy, y_test.npy')
print(f'  {PROC_PATH}X_train_raw.npy, X_test_raw.npy')
print(f'  {PROC_PATH}feature_columns.json')
print(f'  {MODEL_PATH}preprocessor.joblib')